### Black-Scholes framework via Object-oriented programming (OOP)

In [ ]:
import numpy as np
import pandas as pd
import random

from scipy.stats import norm
from scipy.interpolate import interp1d
from scipy.stats import linregress

import plotly.graph_objects as go

In [ ]:
from yahooquery import Ticker

In [ ]:
# data = pd.read_csv('Topic-5_cola_data.csv')
# prices = data['close']
# t = data['date']

<H3>I. Instrument</H3>

First of all, define a financial instrument. Consider a **call option** with payoff function 
$$
f_T = (S_T-K)^+ = max(S_T - K, 0)
$$
$T$ - maturity, $K$ - strike price.

In [ ]:
class option():
    def __init__(self, strike, maturity):
        self.K = strike
        self.T = maturity

    def payoff(self, stock_price):
        pass

class option_call(option):
    def __init__(self, strike, maturity):
        super().__init__(strike, maturity)
    
    def payoff(self, stock_price):
        return max(stock_price - self.K, 0)

In [ ]:
instr = option_call(110, 0.3)
instr.payoff(150)
# instr.payoff(80)

<H3>II. Model</H3>

We use Black-Sholes model with bank account evolution (continuous compounding)
$$
B_t =B_0 e^{rt}
$$
and stock price evolution
$$
S_t = S_0 e^{(\mu - \sigma^2/2)t+\sigma W_t}
$$
In differential form
$$
dS_t = \mu S_tdt+\sigma S_t dW_t
$$
Model parameters: $r$ - risk-free interest rate, $\mu$ - trend, $\sigma$ - volatility.

<H3>III. Parameters estimation (calibration)</H3>

Estimate $r,\mu,\sigma$ from market data. Then identify model and finally calculate fair prices.

In [ ]:
cola = Ticker('KO')
cola_data = cola.history(start='2019-02-19', end = '2020-02-21', interval = '1d')

In [ ]:
t = cola_data.index.get_level_values(level=1)
prices = np.array(cola_data['close'])

In [ ]:
plot = go.Figure()

graph = go.Scatter(x = t, y = prices)
plot.add_trace(graph)

plot.show()

Take Treasury yields
https://www.bloomberg.com/markets/rates-bonds/government-bonds/us

In [ ]:
# time to maturity (in years).
ttm = 4/12

# to estimate r we need discount curve
time_ = [0.25, 0.5, 1, 2, 5, 10, 30]
yield_ = [0.0419, 0.04456, 0.0461, 0.0441, 0.04, 0.0387, 0.0407]
res_func = interp1d(time_, yield_, kind = 'quadratic')
res_func(ttm)

In [ ]:
class discount_curve:
    def __init__(self, times, yields):
        self.interp = interp1d(times, yields, kind = 'quadratic')
   
    def rate(self, years):
        return float(self.interp(years))

In [ ]:
curve = discount_curve(time_, yield_)
curve.rate(ttm)

To estimate trend and volatility we move to the log-prices:
$$
\widetilde{S_t} = ln\left(\frac{S_t}{S_0}\right) = (\mu-\sigma^2/2)t + \sigma W_t,\ t\ge 0
$$
Therefore, $\hat{\sigma} = stdev(\widetilde S),\, \hat{\mu} - \hat{\sigma}^2/2$ = slope of regression line

In [ ]:
class calibration_engine:
    def __init__(self, prices, curve):
        self.data = prices
        self.disc_curve = curve
        
    def estimate_rate(self, ttm):
        return self.disc_curve.rate(ttm)
    
    def estimate_volatility(self):
        return np.std(self.log_data())
    
    def estimate_trend(self):
        n_days = len(self.data)
        t_numer = np.arange(0, n_days)
        logs_ = self.log_data()
        trend = linregress(t_numer, logs_)
        return trend.slope
        
    # utilities
    def log_data(self):
        d0 = self.data[0]
        result = np.log(self.data/d0)
        return result

we are ready to estimatate *risk-free rate*

In [ ]:
estimator = calibration_engine(prices, curve)
r_ = estimator.estimate_rate(ttm) 
# estimator.log_data()
vol_ = estimator.estimate_volatility()
trend_ = estimator.estimate_trend()
print(r_, vol_, trend_)

<H3>II. Model</H3>

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_steps = granularity * max_time
    
    paths = []
    for i in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_steps)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
        
    return np.array(paths)

In [ ]:
class black_scholes:
    def __init__(self, rate, mu_, sigma, initial):
        self.r = rate
        self.mu = mu_
        self.volatility = sigma
        self.s0 = initial
        
    def price_simulations(self, n_paths, T):  
        granul_factor = 100
        W = bm_simulations(n_paths, granul_factor, T)   
        t = np.linspace(0, T, granul_factor*T + 1) 
        trend = self.mu - self.volatility**2/2
        st = [self.s0*np.exp(trend*t + self.volatility*W[i]) for i in range(n_paths)]
                   
        return st        

In [ ]:
mu_ = trend_ + vol_**2/2
model = black_scholes(r_, mu_, vol_, prices[-1])
model.volatility

In [ ]:
n_paths = 10
T_days = int(ttm*365)
paths = model.price_simulations(n_paths, T_days)

In [ ]:
t_total = np.linspace(0, T_days, len(paths[0]))

In [ ]:
plot = go.Figure()

for i in range(0, n_paths):
    graph1 = go.Scatter(x = t_total, y = paths[i])
    plot.add_trace(graph1)
plot.show()

<H3>IV. Pricing</H3>

Correct pricing is done in a model where $r=\mu$ i.e. $dS_t = rS_t dt + \sigma S_t dW_t$
$$
PV = E\left(\frac{f_T}{e^{rT}}\right)=E\left(\frac{(S_T-K)^{+}}{e^{rT}}\right)
$$

Find fair price (PV) of an instrument.

In [ ]:
class pricer:
    def __init__(self, pricing_model):
        self.model = pricing_model
        
    def price(self, instr):
        pass

$$
PV = S_0\Phi(d_+)-e^{-rT}K\Phi(d_{-}),
$$
</br>
$$d_{\pm}=\frac{1}{\sigma\sqrt T}\left(\ln\left(\frac{S_0}{K}\right)+\left(r\pm\frac{\sigma^2}{2}\right)T\right)$$

In [ ]:
class analytic_pricer(pricer):
    def __init__(self, model):
        super().__init__(model)
        
    def price(self, option):
        s0 = self.model.s0
        rate = self.model.r
        sigma = self.model.volatility
        K = option.K
        ttm = option.T
        
        d_plus = (np.log(s0/K)+ttm*(rate+(sigma**2)/2))/(sigma*np.sqrt(ttm))
        d_minus = d_plus-sigma*np.sqrt(ttm)        
        result = s0*norm.cdf(d_plus) - np.exp(-rate*ttm)*K*norm.cdf(d_minus)
        return result

In [ ]:
s0 = prices[-1]
bs_model = black_scholes(r_, r_, vol_, s0)

In [ ]:
s0

In [ ]:
K = s0
# price 4-month European call option
ttm = 4/12
instr = option_call(K, ttm)

In [ ]:
an_pricer = analytic_pricer(bs_model) 
an_price_ = an_pricer.price(instr)
an_price_

In [ ]:
class monte_carlo_pricer(pricer):
    def __init__(self, model, n_paths):
        super().__init__(model)
        self.N = n_paths
        
    def price(self, option):
        ttm = option.T
        norm_rvs = norm.rvs(loc = 0, scale = 1, size = self.N)
        s0 = self.model.s0
        rate = self.model.r
        sigma = self.model.volatility
        K = option.K

        S_T = s0*np.exp(sigma*np.sqrt(ttm)*norm_rvs + (rate - sigma**2/2)*ttm)
        payoffs = [option.payoff(s) for s in S_T] 
        discounted_payoffs = [payoff * np.exp(-rate*ttm) for payoff in payoffs]   
        return np.mean(discounted_payoffs)   

In [ ]:
n_paths = 100_000
mc_pricer = monte_carlo_pricer(bs_model, n_paths)
mc_price_ = mc_pricer.price(instr)
mc_price_

What about price via simulated value from the model i.e. when
$$
S_T = S_0e^{\sigma W_T +(r-\sigma^2/2)T}
$$
is the endpoint of simulated trajectory $(S_t)_{t\ge 0}$?<br/>
Another Monte-Carlo pricer is possible

## Options chains<br>
https://finance.yahoo.com/quote/AAPL/options?p=AAPL<br>
https://finance.yahoo.com/quote/GOOG/options/<br>
https://finance.yahoo.com/quote/KO/options/<br>
Option prices with various strikes and maturities should be used for better pricing. How? Via **Implied Volatility**.

## Numerical methods for differential equations<br>

Consider differential equation and finite difference scheme from Problem 4, exam 2023-24.</br>
Implement finite difference scheme. Output 4 scatterplots (mode=lines) on the same plot: exact solution $y=y(x)$ and 3 solutions $(y_0,\ldots, y_n)$ of finite difference schemes for $n=10,\,20$ and $50$.

In [ ]:
ns = [10, 20, 50]
solution = []
for n in ns:
    h = 1/n
    yn = [2]
    xn = [0]
    for i in range(1, n+1):
        xn.append(i*h)
        yn.append(yn[-1]*(1-2*i*h**2))
    solution.append([xn, yn])

In [ ]:
plot = go.Figure()

x_ = np.linspace(0,1,1001)
y_ = 2*np.exp(-x_**2)
graph = go.Scatter(x = x_, y = y_, name = 'Exact solution')
plot.add_trace(graph)

for i in range(0, len(ns)):
    graph_2 = go.Scatter(x = solution[i][0], y = solution[i][1], name = f'Solution n={ns[i]}')
    plot.add_trace(graph_2)

plot.show()